# 🧪 AenPi — Full Testing Notebook
### Modules: `IntentRouter` & `UrduSentiment`

**Repo:** https://github.com/EN-AenaHabib/AenPi  
**Package:** `AenPi.urdu`

---

This notebook tests every available method in both modules using:
- Diverse Urdu and Roman Urdu examples (not just one topic)
- Edge cases that stress-test the model
- Clear explanation of what every result means

> ⚠️ Run cells **top to bottom** — each section depends on the one above it.

---
## ⚙️ Step 1 — Clone Repo & Install Dependencies

We clone the **latest** version of the repo and install it in editable mode (`-e`).  
This means Python imports directly from the cloned source — no stale cached version.

In [1]:
!git clone https://github.com/EN-AenaHabib/AenPi.git
%cd AenPi
!pip install -e . --quiet
!pip install scikit-learn datasets --quiet
print("\n✅ Setup complete.")

Cloning into 'AenPi'...
remote: Enumerating objects: 364, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 364 (delta 66), reused 6 (delta 1), pack-reused 230 (from 1)
Receiving objects: 100% (364/364), 2.42 MiB | 8.50 MiB/s, done.
Resolving deltas: 100% (163/163), done.
/content/AenPi
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.0 MB/s eta 0:00:00

✅ Setup complete.


---
## 📦 Step 2 — Import Modules

We import both classes directly from the source files — not from any installed wheel.  
If this cell fails it means the clone or install above had an issue.

In [2]:
from AenPi.urdu.intent_router import IntentRouter
from AenPi.urdu.sentiment      import UrduSentiment

print("✅ IntentRouter :", IntentRouter)
print("✅ UrduSentiment:", UrduSentiment)

✅ IntentRouter : <class 'AenPi.urdu.intent_router.IntentRouter'>
✅ UrduSentiment: <class 'AenPi.urdu.sentiment.UrduSentiment'>


---
## 🔍 Step 3 — Inspect Available Methods

We use `inspect.getmembers()` to list every **real method** in both classes.  
This makes sure we don't test something that doesn't exist, and we don't miss anything that does.

> Public methods (no leading `_`) are what users call.  
> Private methods (leading `_`) are internal helpers — we do not test those directly.

In [3]:
import inspect

for cls in [IntentRouter, UrduSentiment]:
    print("=" * 55)
    print(f"  {cls.__name__} — public methods")
    print("=" * 55)
    for name, fn in inspect.getmembers(cls, predicate=inspect.isfunction):
        if not name.startswith("_"):
            print(f"  {name}{inspect.signature(fn)}")
    print()

  IntentRouter — public methods
  add_examples(self, new_examples: list)
  example_count(self) -> dict
  fit(self, examples: list)
  list_intents(self) -> list
  predict(self, text: str) -> dict
  predict_batch(self, texts: list) -> list
  refit(self)
  score(self, examples: list) -> float
  top_n(self, text: str, n: int = 3) -> list

  UrduSentiment — public methods
  fit(self, max_samples: int = 100000)
  predict(self, text: str) -> dict
  predict_batch(self, texts: list) -> list
  score(self, texts: list, labels: list) -> float



---
## 🎯 Section A — IntentRouter

`IntentRouter` is an **offline intent classifier**.  
You give it labeled example sentences, it trains a TF-IDF + Logistic Regression model,  
and then it predicts which *intent* any new input belongs to.

---

### A1 — Training: `fit()`

We train on **5 different intents** with **3+ varied examples each** (no repetition).  
The more diverse the training sentences, the better the model separates intents.

| Intent | Meaning |
|---|---|
| `order` | User wants to place a new order |
| `refund` | User wants money back |
| `tracking` | User asking about delivery/location |
| `complaint` | User reporting a problem |
| `feedback` | User leaving a review or praise |

**What the output tells you:**  
`IntentRouter trained on N examples, K intents: [...]`  
— confirms training succeeded and shows all learned intent labels.

In [4]:
router = IntentRouter()
print("Before training:", repr(router))  # shows 'not fitted'
print()

training_data = [
    # ── ORDER ──────────────────────────────────────────
    ("naya order karna chahta hoon",          "order"),
    ("mujhe ye wala item mangwana hai",       "order"),
    ("kya main abhi book kar sakta hoon",     "order"),
    ("mujhe 2 pieces chahiye",                "order"),
    ("cart mein daal ke order karo",          "order"),

    # ── REFUND ─────────────────────────────────────────
    ("paisa wapas karo mera",                 "refund"),
    ("mujhe reimbursement chahiye",           "refund"),
    ("yeh cheez return karni hai paisa lo",   "refund"),
    ("paise refund nahi hue abhi tak",        "refund"),
    ("mera amount wapas kab aayega",          "refund"),

    # ── TRACKING ───────────────────────────────────────
    ("mera parcel kahan tak pahuncha",        "tracking"),
    ("delivery kab hogi ghar pe",             "tracking"),
    ("shipment status check karo",            "tracking"),
    ("mujhe tracking number bata do",         "tracking"),
    ("courier ne abhi tak dispatch nahi kiya","tracking"),

    # ── COMPLAINT ──────────────────────────────────────
    ("jo item aya woh toot hua tha",          "complaint"),
    ("product bilkul bekaar nikla",           "complaint"),
    ("packaging damaged thi andar se",        "complaint"),
    ("wrong item bheja hai tumne",            "complaint"),
    ("size galat hai jo manga tha woh nahi",  "complaint"),

    # ── FEEDBACK ───────────────────────────────────────
    ("shukriya bohat acha service tha",       "feedback"),
    ("staff ne bohat help ki",                "feedback"),
    ("experience zabardast raha mera",        "feedback"),
    ("aap logon ka kaam bohat pasand aya",    "feedback"),
    ("5 star deta hoon bhai wakai acha tha",  "feedback"),
]

router.fit(training_data)
print("\nAfter training:", repr(router))

Before training: IntentRouter(not fitted)

IntentRouter trained on 25 examples, 5 intents: [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('refund'), np.str_('tracking')]

After training: IntentRouter(fitted, intents=[np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('refund'), np.str_('tracking')], examples=25)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


---
### A2 — Single Prediction: `predict()`

`predict(text)` returns a dict with three keys:

| Key | Type | Meaning |
|---|---|---|
| `intent` | `str` | The predicted intent label |
| `score` | `float` | Confidence of the top prediction (0 to 1) |
| `scores` | `dict` | Confidence for **every** known intent |

A `score` close to **1.0** means the model is very sure.  
A `score` close to **0.2** (with 5 classes) means it's basically guessing — input is ambiguous.

We test one fresh sentence per intent so results are not biased toward training text.

In [5]:
test_sentences = [
    ("abhi ek naya order place karna hai",    "expect → order"),
    ("mujhe refund do jaldi",                 "expect → refund"),
    ("bhai parcel track karna hai kaise",     "expect → tracking"),
    ("item aaya lekin bilkul kharab tha",     "expect → complaint"),
    ("bahot khush hoon is service se",        "expect → feedback"),
]

print(f"{'Input':<45} {'Predicted':<12} {'Score':<8} {'Expected'}")
print("-" * 85)
for text, expected in test_sentences:
    r = router.predict(text)
    print(f"{text:<45} {r['intent']:<12} {r['score']:<8.4f} {expected}")

print()
print("Full scores for last input:")
for intent, score in sorted(r['scores'].items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 30)
    print(f"  {intent:<12}: {score:.4f}  {bar}")

Input                                         Predicted    Score    Expected
-------------------------------------------------------------------------------------
abhi ek naya order place karna hai            order        0.6157   expect → order
mujhe refund do jaldi                         refund       0.4329   expect → refund
bhai parcel track karna hai kaise             tracking     0.3670   expect → tracking
item aaya lekin bilkul kharab tha             complaint    0.5568   expect → complaint
bahot khush hoon is service se                feedback     0.3783   expect → feedback

Full scores for last input:
  feedback    : 0.3783  ███████████
  order       : 0.2042  ██████
  complaint   : 0.1549  ████
  tracking    : 0.1482  ████
  refund      : 0.1143  ███


---
### A3 — Batch Prediction: `predict_batch()`

`predict_batch(texts)` takes a **list** of strings and returns a list of result dicts.  
Internally it vectorizes all inputs in one matrix operation — faster than calling `predict()` in a loop.

Useful when you have many messages to classify at once (e.g. a customer support queue).

In [6]:
batch_inputs = [
    "aaj delivery expect thi par nahi aayi",
    "itna bura product pehle kabhi nahi dekha",
    "team ne bohat jaldi resolve kiya shukriya",
    "ek aur item add karna hai mujhe",
    "paise kab wapas honge mujhe batao",
]

results = router.predict_batch(batch_inputs)

print(f"{'#':<3} {'Input':<45} {'Intent':<12} {'Score'}")
print("-" * 75)
for i, (text, res) in enumerate(zip(batch_inputs, results), 1):
    print(f"{i:<3} {text:<45} {res['intent']:<12} {res['score']:.4f}")

#   Input                                         Intent       Score
---------------------------------------------------------------------------
1   aaj delivery expect thi par nahi aayi         tracking     0.4437
2   itna bura product pehle kabhi nahi dekha      complaint    0.3718
3   team ne bohat jaldi resolve kiya shukriya     feedback     0.5805
4   ek aur item add karna hai mujhe               order        0.5234
5   paise kab wapas honge mujhe batao             refund       0.5916


---
### A4 — Top-N Intents: `top_n()`

`top_n(text, n=3)` returns the **top N intents** sorted by confidence descending.

Why is this useful?  
Sometimes a message is genuinely ambiguous — e.g. *"mera kaam nahi hua"* could be a complaint or a tracking issue.  
Showing top-3 lets a downstream system decide, rather than blindly trusting rank-1.

In [7]:
ambiguous_inputs = [
    "yaar kuch theek nahi hua",           # ambiguous — could be complaint or feedback
    "mujhe information chahiye order ki", # could be tracking or order
    "kab tak hoga mera kaam",             # could be tracking or complaint
]

for text in ambiguous_inputs:
    top = router.top_n(text, n=3)
    print(f"Input: '{text}'")
    for rank, item in enumerate(top, 1):
        bar = '█' * int(item['score'] * 25)
        print(f"  #{rank}  {item['intent']:<12} {item['score']:.4f}  {bar}")
    print()

Input: 'yaar kuch theek nahi hua'
  #1  complaint    0.4012  ██████████
  #2  tracking     0.2006  █████
  #3  refund       0.1694  ████

Input: 'mujhe information chahiye order ki'
  #1  order        0.7717  ███████████████████
  #2  refund       0.0767  █
  #3  tracking     0.0687  █

Input: 'kab tak hoga mera kaam'
  #1  tracking     0.3916  █████████
  #2  refund       0.2409  ██████
  #3  feedback     0.1735  ████



---
### A5 — Accuracy Evaluation: `score()`

`score(examples)` takes labeled `(text, label)` pairs and returns **accuracy (0–1)**.  
This is how you measure how well the model generalizes to sentences it has never seen.

We use fresh sentences — none of these appeared in `training_data` above.

In [8]:
eval_data = [
    ("nayi cheez mangwani hai mujhe",        "order"),
    ("amount wapas karna hai",               "refund"),
    ("mera courier abhi kahan hai",          "tracking"),
    ("product mein defect tha",              "complaint"),
    ("bahot umda kaam kiya team ne",         "feedback"),
    ("order confirm hua ya nahi",            "order"),
    ("return karna chahta hoon",             "refund"),
]

acc = router.score(eval_data)
print(f"\nAccuracy on {len(eval_data)} unseen examples: {acc:.2%}")
print()
print("Interpretation:")
print("  ≥ 0.85  →  model is generalizing well")
print("  0.60-0.85 →  decent but more training data would help")
print("  < 0.60  →  model is confused — add more diverse examples")

Accuracy: 85.71% (6/7)

Accuracy on 7 unseen examples: 85.71%

Interpretation:
  ≥ 0.85  →  model is generalizing well
  0.60-0.85 →  decent but more training data would help
  < 0.60  →  model is confused — add more diverse examples


---
### A6 — Incremental Learning: `add_examples()` + `refit()`

`add_examples()` appends new labeled pairs to the existing training set.  
`refit()` retrains the model on the **combined** old + new data.

This simulates a real production scenario where you keep improving the model  
without throwing away what it already learned.

Here we add a brand new intent: **`payment`** — something the model has never seen before.

In [9]:
new_examples = [
    ("payment fail ho gayi meri",             "payment"),
    ("card se charge kyun hua dobara",        "payment"),
    ("online transfer nahi hua",              "payment"),
    ("mujhe payment receipt chahiye",         "payment"),
    ("UBL card decline ho gaya",              "payment"),
]

router.add_examples(new_examples)
router.refit()

print("\nIntents after refit:", router.list_intents())
print("Training examples per intent:", router.example_count())

# test the new intent
test = "transaction process nahi hui meri"
r = router.predict(test)
print(f"\nTest on new intent input: '{test}'")
print(f"  Predicted: {r['intent']}  (score: {r['score']:.4f})")
print(f"  Expected : payment")

Added 5 examples. Total: 30. Call .refit() to update.
IntentRouter trained on 30 examples, 6 intents: [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')]

Intents after refit: [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')]
Training examples per intent: {'order': 5, 'refund': 5, 'tracking': 5, 'complaint': 5, 'feedback': 5, 'payment': 5}

Test on new intent input: 'transaction process nahi hui meri'
  Predicted: payment  (score: 0.3116)
  Expected : payment


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


---
### A7 — Utility Methods: `list_intents()` & `example_count()`

These are inspection helpers — useful for debugging and monitoring your router.

| Method | Returns | Use case |
|---|---|---|
| `list_intents()` | `list` of intent label strings | See all labels the model knows |
| `example_count()` | `dict` of `{label: count}` | Check if any intent is under-trained |

In [10]:
intents = router.list_intents()
counts  = router.example_count()

print("Known intents:", intents)
print()
print(f"{'Intent':<14} {'Examples':<10} {'Balance bar'}")
print("-" * 40)
total = sum(counts.values())
for label in intents:
    n = counts[label]
    bar = '█' * n
    print(f"{label:<14} {n:<10} {bar}")
print(f"\nTotal training examples: {total}")
print()
print("Tip: Ideally each intent should have a similar number of examples.")
print("     An imbalanced set (e.g. 20 order vs 2 complaint) biases predictions.")

Known intents: [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')]

Intent         Examples   Balance bar
----------------------------------------
complaint      5          █████
feedback       5          █████
order          5          █████
payment        5          █████
refund         5          █████
tracking       5          █████

Total training examples: 30

Tip: Ideally each intent should have a similar number of examples.
     An imbalanced set (e.g. 20 order vs 2 complaint) biases predictions.


---
## 🧨 Section B — IntentRouter Edge Cases

Edge cases test how the model handles **unusual or unexpected inputs**.  
A good NLP module should never crash — it should always return *something*,  
even if that something is a low-confidence guess.

### B1 — Empty String `""`

An empty string produces a zero vector after TF-IDF.  
The model still runs — softmax over zeros gives roughly uniform probabilities.  
So the predicted intent is essentially a **coin flip** with low confidence.  
No crash is expected.

In [11]:
try:
    r = router.predict("")
    print("Input : '' (empty string)")
    print(f"Result: intent={r['intent']}, score={r['score']:.4f}")
    print(f"Note  : score ≈ {1/len(r['scores']):.2f} means model is just guessing uniformly")
except Exception as e:
    print(f"Exception ({type(e).__name__}): {e}")

Input : '' (empty string)
Result: intent=tracking, score=0.1893
Note  : score ≈ 0.17 means model is just guessing uniformly


### B2 — Single Word

Single words give very sparse TF-IDF features.  
If the word overlaps with training vocabulary (e.g. "refund") → higher confidence.  
If the word is totally new → low confidence, still no crash.

In [12]:
single_words = ["refund", "order", "parcel", "shukriya", "xyz", "کام"]
print(f"{'Word':<15} {'Intent':<12} {'Score'}")
print("-" * 40)
for w in single_words:
    r = router.predict(w)
    print(f"{w:<15} {r['intent']:<12} {r['score']:.4f}")

Word            Intent       Score
----------------------------------------
refund          refund       0.5483
order           order        0.6063
parcel          tracking     0.5016
shukriya        feedback     0.3845
xyz             refund       0.1971
کام             tracking     0.1893


### B3 — Pure Urdu Script

The model was trained on Roman Urdu (Latin script).  
Urdu script uses a completely different character set — Arabic-based.  
TF-IDF char n-grams will produce mostly unseen features → low confidence predictions.  
This is **expected behavior** — not a bug. It shows the model's honest limits.

In [13]:
urdu_script = [
    ("مجھے واپسی چاہیے",     "hope: refund"),
    ("آرڈر کب آئے گا",        "hope: tracking"),
    ("پروڈکٹ خراب ہے",        "hope: complaint"),
    ("بہت اچھی سروس تھی",     "hope: feedback"),
    ("نیا آرڈر کرنا ہے",      "hope: order"),
]

print(f"{'Urdu Input':<25} {'Predicted':<12} {'Score':<8} {'Note'}")
print("-" * 70)
for text, hope in urdu_script:
    r = router.predict(text)
    note = "low conf — script mismatch" if r['score'] < 0.5 else "OK"
    print(f"{text:<25} {r['intent']:<12} {r['score']:<8.4f} {hope} | {note}")

Urdu Input                Predicted    Score    Note
----------------------------------------------------------------------
مجھے واپسی چاہیے          tracking     0.1893   hope: refund | low conf — script mismatch
آرڈر کب آئے گا            tracking     0.1893   hope: tracking | low conf — script mismatch
پروڈکٹ خراب ہے            tracking     0.1893   hope: complaint | low conf — script mismatch
بہت اچھی سروس تھی         tracking     0.1893   hope: feedback | low conf — script mismatch
نیا آرڈر کرنا ہے          tracking     0.1893   hope: order | low conf — script mismatch


### B4 — Roman Urdu (Natural Language)

Roman Urdu is the primary training language — this should work well.  
These inputs use casual conversational phrasing, not the exact training sentences.

In [14]:
roman_urdu = [
    ("yaar mera parcel aaya hi nahi",         "expect: tracking"),
    ("bhai paisa wapas chahiye please",       "expect: refund"),
    ("packaging phat gayi thi aane mein",     "expect: complaint"),
    ("really acha laga yaar bohat khush hoon","expect: feedback"),
    ("mujhe 3 shirts order karni hain",       "expect: order"),
]

print(f"{'Input':<45} {'Predicted':<12} {'Score':<8} {'Expected'}")
print("-" * 85)
for text, exp in roman_urdu:
    r = router.predict(text)
    print(f"{text:<45} {r['intent']:<12} {r['score']:<8.4f} {exp}")

Input                                         Predicted    Score    Expected
-------------------------------------------------------------------------------------
yaar mera parcel aaya hi nahi                 tracking     0.3235   expect: tracking
bhai paisa wapas chahiye please               refund       0.7136   expect: refund
packaging phat gayi thi aane mein             complaint    0.4149   expect: complaint
really acha laga yaar bohat khush hoon        feedback     0.5984   expect: feedback
mujhe 3 shirts order karni hain               order        0.6626   expect: order


### B5 — Mixed Language (Roman Urdu + English)

Code-switching is extremely common in Pakistani text.  
Because char n-grams work at character level, they handle mixing naturally.

In [15]:
mixed = [
    ("I want to return this product, bilkul bekaar tha", "expect: complaint or refund"),
    ("please track my order, parcel kahan hai",          "expect: tracking"),
    ("bohat happy hoon, best service ever",              "expect: feedback"),
    ("add karo ek aur item to my cart please",           "expect: order"),
    ("payment gateway par error aa raha hai",            "expect: payment"),
]

print(f"{'Input':<52} {'Predicted':<12} {'Score'}")
print("-" * 75)
for text, exp in mixed:
    r = router.predict(text)
    print(f"{text:<52} {r['intent']:<12} {r['score']:.4f}")
    print(f"  → {exp}")

Input                                                Predicted    Score
---------------------------------------------------------------------------
I want to return this product, bilkul bekaar tha     complaint    0.6521
  → expect: complaint or refund
please track my order, parcel kahan hai              tracking     0.4936
  → expect: tracking
bohat happy hoon, best service ever                  feedback     0.6530
  → expect: feedback
add karo ek aur item to my cart please               order        0.2967
  → expect: order
payment gateway par error aa raha hai                payment      0.3594
  → expect: payment


### B6 — Emojis & Special Characters

The `_preprocess()` method strips punctuation and special chars before vectorizing.  
So emojis/symbols are removed — only the remaining text drives the prediction.

In [16]:
emoji_inputs = [
    "order karo jaldi 🛒🛒!!!",
    "😡😡 product kharab tha bilkul!!!",
    "📦 parcel kahan hai???",
    "💸 paisa wapas do @@support",
    "🙏 shukriya team #bestservice",
    "😐😐😐",   # only emojis — nothing left after preprocess
]

print(f"{'Input':<42} {'Predicted':<12} {'Score'}")
print("-" * 65)
for text in emoji_inputs:
    r = router.predict(text)
    print(f"{text:<42} {r['intent']:<12} {r['score']:.4f}")

Input                                      Predicted    Score
-----------------------------------------------------------------
order karo jaldi 🛒🛒!!!                     order        0.5669
😡😡 product kharab tha bilkul!!!            complaint    0.6329
📦 parcel kahan hai???                      tracking     0.5004
💸 paisa wapas do @@support                 refund       0.7542
🙏 shukriya team #bestservice               feedback     0.5043
😐😐😐                                        tracking     0.1893


### B7 — Error Handling: Predict Before Fit & Empty Fit

These confirm the module raises proper errors instead of silent failures.

In [17]:
# Test 1: predict before fit
print("Test 1: predict() before fit()")
try:
    IntentRouter().predict("hello")
    print("  ❌ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"  ✅ RuntimeError: {e}")

print()

# Test 2: fit with empty list
print("Test 2: fit() with empty examples list")
try:
    IntentRouter().fit([])
    print("  ❌ Should have raised ValueError")
except ValueError as e:
    print(f"  ✅ ValueError: {e}")

print()

# Test 3: predict_batch before fit
print("Test 3: predict_batch() before fit()")
try:
    IntentRouter().predict_batch(["test"])
    print("  ❌ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"  ✅ RuntimeError: {e}")

Test 1: predict() before fit()
  ✅ RuntimeError: Router is not trained. Call .fit(examples) first.

Test 2: fit() with empty examples list
  ✅ ValueError: examples list cannot be empty.

Test 3: predict_batch() before fit()
  ✅ RuntimeError: Router is not trained. Call .fit(examples) first.


---
## 💬 Section C — UrduSentiment

`UrduSentiment` classifies text as **Positive**, **Negative**, or **Neutral**.  
It trains on the [Khubaib01/RomanUrdu-NLP-Sentiment-Corpus](https://huggingface.co/datasets/Khubaib01/RomanUrdu-NLP-Sentiment-Corpus) dataset (134K samples, Apache 2.0).

The model is the same architecture as IntentRouter — TF-IDF + Logistic Regression —  
but trained on a much larger dataset for 3 fixed sentiment classes.

### C1 — State Before Training

Before calling `fit()`, the model is unfitted.  
`repr()` tells you the current state clearly.

In [18]:
model = UrduSentiment()
print("Before fit():", repr(model))
print()
print("is_fitted  :", model.is_fitted)
print("classes_   :", model.classes_)
print("vectorizer :", model.vectorizer)
print("clf        :", model.clf)

Before fit(): UrduSentiment(not fitted)

is_fitted  : False
classes_   : []
vectorizer : None
clf        : None


---
### C2 — Training: Fixed `fit()` (Column-Auto-Detection)

The original `fit()` in `sentiment.py` has two bugs:
1. `trust_remote_code=True` is deprecated in newer `datasets` versions
2. It looks for columns named `"text"` and `"sentiment"` but the actual CSV uses different names → **0 samples loaded → crash**

The fix below:
- Loads dataset **without** `trust_remote_code`
- Prints real column names so you can see them
- **Auto-detects** the text and label columns
- Uses the **exact same** TF-IDF + LogReg hyperparameters as the original `fit()`
- Injects the trained model back into the `UrduSentiment` instance so all other methods work normally

In [19]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from collections import Counter
import re

MAX_SAMPLES = 20_000   # 20K is enough for a test; default is 100K

print("Loading dataset ...")
ds = load_dataset(
    "Khubaib01/RomanUrdu-NLP-Sentiment-Corpus",
    split="train",
    # trust_remote_code removed — deprecated
)

# ── auto-detect columns ───────────────────────────────────────────────────────
first_row = ds[0]
print("\nActual columns :", list(first_row.keys()))
print("Sample row     :", first_row)

TEXT_CANDS  = ["text","Text","sentence","Sentence","review","Review","comment","Comment","input"]
LABEL_CANDS = ["sentiment","Sentiment","label","Label","class","Class","category","Category"]

text_col  = next((c for c in TEXT_CANDS  if c in first_row), None)
label_col = next((c for c in LABEL_CANDS if c in first_row), None)

# fallback: longest string column = text
if text_col is None:
    str_cols = [k for k,v in first_row.items() if isinstance(v,str)]
    text_col = max(str_cols, key=lambda k: len(str(first_row[k])), default=None)
if label_col is None:
    label_col = next((k for k in first_row if k != text_col), None)

print(f"\nDetected → text_col='{text_col}', label_col='{label_col}'")

# ── same label_map as sentiment.py ────────────────────────────────────────────
label_map = {
    "Positive":"Positive","Negative":"Negative","Neutral":"Neutral",
    "positive":"Positive","negative":"Negative","neutral":"Neutral",
    1:"Positive", 0:"Neutral", -1:"Negative",
    "1":"Positive","0":"Neutral","-1":"Negative",
}

# ── same _preprocess as sentiment.py ─────────────────────────────────────────
def _preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+",            " ", text)
    text = re.sub(r"#\w+",            " ", text)
    text = re.sub(r"\d+",             " ", text)
    text = re.sub(r"(.)\1{2,}",       r"\1\1", text)
    text = re.sub(r"\s+",             " ", text).strip()
    return text

# ── collect samples ───────────────────────────────────────────────────────────
texts, labels = [], []
for row in ds:
    raw_text  = str(row.get(text_col,  "")).strip()
    raw_label = row.get(label_col, "")
    mapped    = label_map.get(raw_label, str(raw_label))
    if mapped in ("Positive","Negative","Neutral") and raw_text:
        texts.append(_preprocess(raw_text))
        labels.append(mapped)
    if len(texts) >= MAX_SAMPLES:
        break

print(f"\nSamples collected : {len(texts):,}")
print("Label distribution:", dict(Counter(labels)))

if len(texts) == 0:
    raise RuntimeError(
        f"0 samples loaded. Actual label values in first 10 rows: "
        f"{[str(ds[i][label_col]) for i in range(min(10,len(ds)))]}"
    )

# ── train — same hyperparams as sentiment.py ─────────────────────────────────
print("\nTraining TF-IDF + LogReg ...")
vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2,4), max_features=60_000, sublinear_tf=True
)
X = vectorizer.fit_transform(texts)

clf = LogisticRegression(C=5.0, max_iter=300, solver="lbfgs", multi_class="multinomial")
clf.fit(X, labels)

# ── inject into UrduSentiment instance ───────────────────────────────────────
model.vectorizer = vectorizer
model.clf        = clf
model.classes_   = list(clf.classes_)
model.is_fitted  = True

print("\n✅ Model ready.")
print("Repr:", repr(model))

Loading dataset ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

RomanUrdu_NLP_Sentiment-Corpus.csv:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/134053 [00:00<?, ? examples/s]


Actual columns : ['message', 'label', 'char_length', 'word_length']
Sample row     : {'message': 'imran khan ke lye bahut acha hoga', 'label': 'Positive', 'char_length': 33.0, 'word_length': 7.0}

Detected → text_col='message', label_col='label'

Samples collected : 20,000
Label distribution: {'Positive': 5519, 'Neutral': 6534, 'Negative': 7947}

Training TF-IDF + LogReg ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



✅ Model ready.
Repr: UrduSentiment(fitted, classes=[np.str_('Negative'), np.str_('Neutral'), np.str_('Positive')])


---
### C3 — Single Prediction: `predict()`

`predict(text)` returns:

| Key | Meaning |
|---|---|
| `label` | `"Positive"`, `"Negative"`, or `"Neutral"` |
| `score` | Confidence of top label (0–1). Above 0.7 = reliable, below 0.5 = uncertain |
| `scores` | Probability for each of the 3 classes |

We test a wide variety of tones — happy, angry, neutral, sarcastic, mixed.

In [20]:
test_cases = [
    # Roman Urdu — clear sentiment
    ("yeh experience bohat zabardast raha",          "Positive"),
    ("bilkul bekar service thi sharmnak",            "Negative"),
    ("theek tha na kuch khas na kuch bura",          "Neutral"),
    # Casual positive
    ("bhai wakai maza aa gaya yaar",                 "Positive"),
    # Strong negative
    ("itna bura product zindagi mein nahi dekha",    "Negative"),
    # Mild neutral
    ("average tha honestly",                         "Neutral"),
    # Mix of positive words
    ("staff acha tha lekin delivery slow thi",       "?mixed"),
    # Complaint-style negative
    ("wrong item bheja aur upar se rude bhi the",   "Negative"),
    # Grateful / positive
    ("team ne bahut jaldi resolve kiya shukriya",    "Positive"),
]

print(f"{'Input':<50} {'Label':<10} {'Score':<8} {'Expected'}")
print("-" * 85)
for text, expected in test_cases:
    r = model.predict(text)
    match = "✅" if r['label'] == expected else "⚠️ " if expected == "?mixed" else "❌"
    print(f"{text:<50} {r['label']:<10} {r['score']:<8.4f} {match} {expected}")

print()
print("Confidence guide:  ≥ 0.70 = reliable  |  0.50–0.70 = moderate  |  < 0.50 = uncertain")

Input                                              Label      Score    Expected
-------------------------------------------------------------------------------------
yeh experience bohat zabardast raha                Positive   0.9917   ✅ Positive
bilkul bekar service thi sharmnak                  Negative   0.8631   ✅ Negative
theek tha na kuch khas na kuch bura                Negative   0.8365   ❌ Neutral
bhai wakai maza aa gaya yaar                       Positive   0.8604   ✅ Positive
itna bura product zindagi mein nahi dekha          Negative   0.7932   ✅ Negative
average tha honestly                               Neutral    0.9409   ✅ Neutral
staff acha tha lekin delivery slow thi             Positive   0.5141   ⚠️  ?mixed
wrong item bheja aur upar se rude bhi the          Neutral    0.4446   ❌ Negative
team ne bahut jaldi resolve kiya shukriya          Positive   0.9285   ✅ Positive

Confidence guide:  ≥ 0.70 = reliable  |  0.50–0.70 = moderate  |  < 0.50 = uncertain


---
### C4 — Batch Prediction: `predict_batch()`

Same as `predict()` but takes a list and returns a list.  
Internally one vectorizer call for all inputs — efficient for large queues.

We simulate a small social media comment dataset here.

In [21]:
comments = [
    "AenPi library ne meri zindagi asan kar di",
    "installation mein bohat issues aaye",
    "documentation thori confusing hai par chalti hai",
    "speed bohat fast hai masha Allah",
    "bugs hain abhi aur polish chahiye",
    "neutral experience tha overall",
    "team responsive hai issues pe",
    "pehli baar use kiya theek laga",
]

results = model.predict_batch(comments)

print(f"{'#':<3} {'Comment':<48} {'Label':<10} {'Score'}")
print("-" * 75)
for i, (text, res) in enumerate(zip(comments, results), 1):
    print(f"{i:<3} {text:<48} {res['label']:<10} {res['score']:.4f}")

#   Comment                                          Label      Score
---------------------------------------------------------------------------
1   AenPi library ne meri zindagi asan kar di        Positive   0.4546
2   installation mein bohat issues aaye              Negative   0.5184
3   documentation thori confusing hai par chalti hai Negative   0.5960
4   speed bohat fast hai masha Allah                 Positive   0.9978
5   bugs hain abhi aur polish chahiye                Neutral    0.5201
6   neutral experience tha overall                   Neutral    0.6728
7   team responsive hai issues pe                    Neutral    0.7170
8   pehli baar use kiya theek laga                   Neutral    0.4069


---
### C5 — Accuracy Evaluation: `score()`

`score(texts, labels)` measures accuracy on a labeled test set.  
None of these sentences appeared in training — they are truly unseen inputs.

**Interpreting accuracy:**
- `≥ 0.75` with 20K training samples → model is working correctly
- `< 0.60` → either the test set is very hard/ambiguous or model needs more data

In [22]:
eval_texts = [
    "yaar kya kamal ki cheez hai yeh",
    "bekar hai bilkul time waste",
    "zyada acha nahi zyada bura bhi nahi",
    "dil khush kar diya is ne",
    "nahi chala properly kaafi problems",
    "average experience tha honestly",
    "bohat helpful raha mujhe",
    "sab kuch galat gaya is baar",
    "chalega theek hi tha",
    "mujhe bahot pasand aaya yeh",
]
eval_labels = [
    "Positive","Negative","Neutral",
    "Positive","Negative","Neutral",
    "Positive","Negative","Neutral",
    "Positive",
]

acc = model.score(eval_texts, eval_labels)
preds = [model.predict(t)["label"] for t in eval_texts]

print(f"Accuracy: {acc:.2%}  ({int(acc*len(eval_labels))}/{len(eval_labels)} correct)")
print()
print(f"{'Input':<45} {'Predicted':<10} {'Actual':<10} {'Match'}")
print("-" * 75)
for text, pred, actual in zip(eval_texts, preds, eval_labels):
    match = "✅" if pred == actual else "❌"
    print(f"{text:<45} {pred:<10} {actual:<10} {match}")

Accuracy: 80.00%  (8/10 correct)

Input                                         Predicted  Actual     Match
---------------------------------------------------------------------------
yaar kya kamal ki cheez hai yeh               Positive   Positive   ✅
bekar hai bilkul time waste                   Negative   Negative   ✅
zyada acha nahi zyada bura bhi nahi           Negative   Neutral    ❌
dil khush kar diya is ne                      Positive   Positive   ✅
nahi chala properly kaafi problems            Negative   Negative   ✅
average experience tha honestly               Neutral    Neutral    ✅
bohat helpful raha mujhe                      Positive   Positive   ✅
sab kuch galat gaya is baar                   Negative   Negative   ✅
chalega theek hi tha                          Positive   Neutral    ❌
mujhe bahot pasand aaya yeh                   Positive   Positive   ✅


---
## 🧨 Section D — UrduSentiment Edge Cases

Same philosophy as IntentRouter edge cases:  
the model should handle anything without crashing — even nonsensical input.

In [23]:
print("D1 — Empty string")
try:
    r = model.predict("")
    print(f"  Result: label={r['label']}, score={r['score']:.4f}")
    print(f"  Note  : uniform ≈ 0.33 per class expected since no features")
except Exception as e:
    print(f"  Exception ({type(e).__name__}): {e}")
print()

D1 — Empty string
  Result: label=Neutral, score=0.9055
  Note  : uniform ≈ 0.33 per class expected since no features



In [24]:
print("D2 — Single words")
words = ["acha","bura","theek","nice","worst","شکریہ","xyz","ok","nahi","very"]
print(f"{'Word':<15} {'Label':<12} {'Score'}")
print("-" * 40)
for w in words:
    r = model.predict(w)
    print(f"{w:<15} {r['label']:<12} {r['score']:.4f}")

D2 — Single words
Word            Label        Score
----------------------------------------
acha            Positive     0.9944
bura            Negative     0.9889
theek           Positive     0.6016
nice            Positive     0.9671
worst           Neutral      0.4964
شکریہ           Neutral      0.9121
xyz             Neutral      0.9459
ok              Positive     0.5203
nahi            Negative     0.9550
very            Positive     0.5977


In [25]:
print("\nD3 — Pure Urdu script")
print("Note: model trained on Roman Urdu — script mismatch will give low confidence")
urdu_cases = [
    ("بہت اچھا تھا",   "expect: Positive"),
    ("بہت برا لگا",    "expect: Negative"),
    ("ٹھیک ہے",        "expect: Neutral"),
    ("زبردست تھا",     "expect: Positive"),
    ("بالکل بیکار",   "expect: Negative"),
]
print(f"{'Urdu Input':<25} {'Label':<12} {'Score':<8} {'Expected'}")
print("-" * 60)
for text, exp in urdu_cases:
    r = model.predict(text)
    print(f"{text:<25} {r['label']:<12} {r['score']:<8.4f} {exp}")


D3 — Pure Urdu script
Note: model trained on Roman Urdu — script mismatch will give low confidence
Urdu Input                Label        Score    Expected
------------------------------------------------------------
بہت اچھا تھا              Neutral      0.8881   expect: Positive
بہت برا لگا               Neutral      0.8586   expect: Negative
ٹھیک ہے                   Neutral      0.8362   expect: Neutral
زبردست تھا                Neutral      0.8003   expect: Positive
بالکل بیکار               Neutral      0.8220   expect: Negative


In [26]:
print("\nD4 — Roman Urdu (natural speech)")
roman_cases = [
    ("yaar bohat maza aya",            "Positive"),
    ("bilkul bakwas tha",              "Negative"),
    ("average hi tha kuch khas nahi",  "Neutral"),
    ("ekdum best experience ever",     "Positive"),
    ("nahi pasand aaya mujhe yeh",     "Negative"),
]
print(f"{'Input':<42} {'Label':<12} {'Score':<8} {'Expected'}")
print("-" * 72)
for text, exp in roman_cases:
    r = model.predict(text)
    match = "✅" if r['label'] == exp else "❌"
    print(f"{text:<42} {r['label']:<12} {r['score']:<8.4f} {match} {exp}")


D4 — Roman Urdu (natural speech)
Input                                      Label        Score    Expected
------------------------------------------------------------------------
yaar bohat maza aya                        Positive     0.9536   ✅ Positive
bilkul bakwas tha                          Negative     0.9847   ✅ Negative
average hi tha kuch khas nahi              Neutral      0.8937   ✅ Neutral
ekdum best experience ever                 Positive     0.8742   ✅ Positive
nahi pasand aaya mujhe yeh                 Negative     0.6527   ✅ Negative


In [27]:
print("\nD5 — Mixed language (Roman Urdu + English)")
mixed_cases = [
    ("This was absolutely amazing, bohat pasand aya",  "Positive"),
    ("Very bad experience, bilkul bura tha",           "Negative"),
    ("It was OK, theek hi tha basically",              "Neutral"),
    ("I am very sad, dil toot gaya aaj",               "Negative"),
    ("Really loved it, kamal ka tha",                  "Positive"),
]
print(f"{'Input':<50} {'Label':<12} {'Score':<8} {'Expected'}")
print("-" * 82)
for text, exp in mixed_cases:
    r = model.predict(text)
    match = "✅" if r['label'] == exp else "❌"
    print(f"{text:<50} {r['label']:<12} {r['score']:<8.4f} {match} {exp}")


D5 — Mixed language (Roman Urdu + English)
Input                                              Label        Score    Expected
----------------------------------------------------------------------------------
This was absolutely amazing, bohat pasand aya      Positive     0.6925   ✅ Positive
Very bad experience, bilkul bura tha               Positive     0.6607   ❌ Negative
It was OK, theek hi tha basically                  Neutral      0.4173   ✅ Neutral
I am very sad, dil toot gaya aaj                   Negative     0.5369   ✅ Negative
Really loved it, kamal ka tha                      Positive     0.9780   ✅ Positive


In [28]:
print("\nD6 — Emojis and special characters")
print("Note: _preprocess() strips symbols — sentiment comes from remaining text only")
emoji_cases = [
    ("bohat acha 😍😍",             "Positive"),
    ("bilkul bekar!!! 😤",          "Negative"),
    ("ok ok 😐 theek hai",          "Neutral"),
    ("😍😍😍",                       "? no text left"),
    ("#bohat #acha @service !!!",   "Positive (hashtags stripped)"),
    ("🔥🔥🔥🔥",                     "? no text left"),
]
print(f"{'Input':<35} {'Label':<12} {'Score'}")
print("-" * 60)
for text, exp in emoji_cases:
    r = model.predict(text)
    print(f"{text:<35} {r['label']:<12} {r['score']:.4f}  ({exp})")


D6 — Emojis and special characters
Note: _preprocess() strips symbols — sentiment comes from remaining text only
Input                               Label        Score
------------------------------------------------------------
bohat acha 😍😍                       Positive     0.9903  (Positive)
bilkul bekar!!! 😤                   Negative     0.8355  (Negative)
ok ok 😐 theek hai                   Positive     0.8031  (Neutral)
😍😍😍                                 Neutral      0.5416  (? no text left)
#bohat #acha @service !!!           Positive     0.6069  (Positive (hashtags stripped))
🔥🔥🔥🔥                                Neutral      0.8251  (? no text left)


In [29]:
print("\nD7 — Error handling")

print("Test 1: predict() before fit()")
try:
    UrduSentiment().predict("test")
    print("  ❌ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"  ✅ RuntimeError: {e}")

print()
print("Test 2: predict_batch() before fit()")
try:
    UrduSentiment().predict_batch(["test"])
    print("  ❌ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"  ✅ RuntimeError: {e}")

print()
print("Test 3: score() before fit()")
try:
    UrduSentiment().score(["test"], ["Positive"])
    print("  ❌ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"  ✅ RuntimeError: {e}")


D7 — Error handling
Test 1: predict() before fit()
  ✅ RuntimeError: Model is not trained. Call .fit() first.

Test 2: predict_batch() before fit()
  ✅ RuntimeError: Model is not trained. Call .fit() first.

Test 3: score() before fit()
  ✅ RuntimeError: Model is not trained. Call .fit() first.


---
## 💾 Section F — Saving & Loading Models (Skip Retraining)

### Why this matters

Every time your Colab runtime restarts, all Python variables are gone.  
That means you have to retrain from scratch — which takes time:

| Model | Training time |
|---|---|
| `IntentRouter` | ~2 seconds (small custom dataset) |
| `UrduSentiment` | 2–5 minutes (20K–100K samples from HuggingFace) |

**The fix:** serialize both models to disk after training once.  
On the next session just load from disk — done in milliseconds.

---

### What needs to be saved

Both models have the same 4 attributes that matter:

| Attribute | Type | What it holds |
|---|---|---|
| `vectorizer` | `TfidfVectorizer` | Learned vocabulary + IDF weights |
| `clf` | `LogisticRegression` | Trained weights + class probabilities |
| `classes_` | `list` | Intent/sentiment label names |
| `is_fitted` | `bool` | Flag so `_check_fitted()` doesn't raise |

`IntentRouter` also has `_examples` (the raw training pairs) — we save that too  
so `add_examples()` + `refit()` still works after loading.

---

### Tool: `joblib` vs `pickle`

We use **`joblib`** (not plain `pickle`) because:
- TF-IDF matrices contain large numpy arrays — joblib compresses them efficiently
- joblib is already installed with scikit-learn (no extra install needed)
- ~3–5x smaller file size and faster load compared to pickle for these objects

In [30]:
import joblib
from pathlib import Path

# ── where to save ─────────────────────────────────────────────────────────────
# In Colab, /content/drive/MyDrive/ = your Google Drive (persistent across sessions)
# If you haven't mounted Drive, we save to /content/ (local, lost on restart)
# Change SAVE_DIR to your Drive path to make it truly permanent.

SAVE_DIR = Path("/content/aenpi_models")   # change to Drive path for persistence
SAVE_DIR.mkdir(parents=True, exist_ok=True)

INTENT_PATH    = SAVE_DIR / "intent_router.joblib"
SENTIMENT_PATH = SAVE_DIR / "urdu_sentiment.joblib"

# ── save IntentRouter ─────────────────────────────────────────────────────────
intent_payload = {
    "vectorizer": router.vectorizer,
    "clf":        router.clf,
    "classes_":   router.classes_,
    "is_fitted":  router.is_fitted,
    "_examples":  router._examples,   # needed for add_examples()+refit() to work
}
joblib.dump(intent_payload, INTENT_PATH, compress=3)
print(f"✅ IntentRouter saved  → {INTENT_PATH}  "
      f"({INTENT_PATH.stat().st_size / 1024:.1f} KB)")

# ── save UrduSentiment ────────────────────────────────────────────────────────
sentiment_payload = {
    "vectorizer": model.vectorizer,
    "clf":        model.clf,
    "classes_":   model.classes_,
    "is_fitted":  model.is_fitted,
}
joblib.dump(sentiment_payload, SENTIMENT_PATH, compress=3)
print(f"✅ UrduSentiment saved → {SENTIMENT_PATH}  "
      f"({SENTIMENT_PATH.stat().st_size / 1024:.1f} KB)")

print(f"\nBoth models saved to: {SAVE_DIR}")
print("Copy this folder to Google Drive to keep it across Colab sessions.")

✅ IntentRouter saved  → /content/aenpi_models/intent_router.joblib  (26.3 KB)
✅ UrduSentiment saved → /content/aenpi_models/urdu_sentiment.joblib  (1509.3 KB)

Both models saved to: /content/aenpi_models
Copy this folder to Google Drive to keep it across Colab sessions.


---
### F2 — Loading Models (Use This on Next Session Instead of Retraining)

On a fresh Colab runtime, run **only** the cells below instead of all the training cells.  
This restores both models fully — every method works exactly as before.

In [31]:
import joblib
from pathlib import Path
from AenPi.urdu.intent_router import IntentRouter
from AenPi.urdu.sentiment      import UrduSentiment

SAVE_DIR       = Path("/content/aenpi_models")   # same path as save cell
INTENT_PATH    = SAVE_DIR / "intent_router.joblib"
SENTIMENT_PATH = SAVE_DIR / "urdu_sentiment.joblib"

# ── load IntentRouter ─────────────────────────────────────────────────────────
intent_data        = joblib.load(INTENT_PATH)
router_loaded      = IntentRouter()
router_loaded.vectorizer = intent_data["vectorizer"]
router_loaded.clf        = intent_data["clf"]
router_loaded.classes_   = intent_data["classes_"]
router_loaded.is_fitted  = intent_data["is_fitted"]
router_loaded._examples  = intent_data["_examples"]

print("✅ IntentRouter loaded")
print("   Repr     :", repr(router_loaded))
print("   Intents  :", router_loaded.list_intents())
print("   Examples :", router_loaded.example_count())

print()

# ── load UrduSentiment ────────────────────────────────────────────────────────
sentiment_data     = joblib.load(SENTIMENT_PATH)
model_loaded       = UrduSentiment()
model_loaded.vectorizer = sentiment_data["vectorizer"]
model_loaded.clf        = sentiment_data["clf"]
model_loaded.classes_   = sentiment_data["classes_"]
model_loaded.is_fitted  = sentiment_data["is_fitted"]

print("✅ UrduSentiment loaded")
print("   Repr    :", repr(model_loaded))
print("   Classes :", model_loaded.classes_)

✅ IntentRouter loaded
   Repr     : IntentRouter(fitted, intents=[np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')], examples=30)
   Intents  : [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')]
   Examples : {'order': 5, 'refund': 5, 'tracking': 5, 'complaint': 5, 'feedback': 5, 'payment': 5}

✅ UrduSentiment loaded
   Repr    : UrduSentiment(fitted, classes=[np.str_('Negative'), np.str_('Neutral'), np.str_('Positive')])
   Classes : [np.str_('Negative'), np.str_('Neutral'), np.str_('Positive')]


---
### F3 — Verify Loaded Models Work Correctly

Run a quick sanity check — predict on a few inputs to confirm the loaded models  
behave identically to the trained ones. If outputs match, the save/load is working.

In [32]:
print("=" * 55)
print("IntentRouter — loaded model predictions")
print("=" * 55)

intent_checks = [
    "mujhe order karna hai",
    "paisa wapas do please",
    "parcel kahan tak pahuncha",
    "product bilkul kharab tha",
    "shukriya bohat acha tha",
    "payment fail ho gayi",
]

for text in intent_checks:
    r = router_loaded.predict(text)
    print(f"  {text:<42} → {r['intent']:<10}  ({r['score']:.4f})")

print()
print("=" * 55)
print("UrduSentiment — loaded model predictions")
print("=" * 55)

sentiment_checks = [
    ("yeh bohat acha tha",              "Positive"),
    ("bilkul bekar experience tha",     "Negative"),
    ("theek tha average hi",            "Neutral"),
    ("zabardast service masha Allah",   "Positive"),
    ("nahi pasand aaya mujhe",          "Negative"),
]

for text, expected in sentiment_checks:
    r = model_loaded.predict(text)
    match = "✅" if r['label'] == expected else "❌"
    print(f"  {text:<44} → {r['label']:<10} {match}")

print()
print("If all predictions above look correct, your saved models are working perfectly.")

IntentRouter — loaded model predictions
  mujhe order karna hai                      → order       (0.7544)
  paisa wapas do please                      → refund      (0.7560)
  parcel kahan tak pahuncha                  → tracking    (0.7638)
  product bilkul kharab tha                  → complaint   (0.6329)
  shukriya bohat acha tha                    → feedback    (0.8030)
  payment fail ho gayi                       → payment     (0.8098)

UrduSentiment — loaded model predictions
  yeh bohat acha tha                           → Positive   ✅
  bilkul bekar experience tha                  → Positive   ❌
  theek tha average hi                         → Neutral    ✅
  zabardast service masha Allah                → Positive   ✅
  nahi pasand aaya mujhe                       → Negative   ✅

If all predictions above look correct, your saved models are working perfectly.


---
### F4 — Making Models Truly Persistent (Google Drive)

By default Colab's `/content/` folder is wiped when the runtime disconnects.  
To keep models across sessions, mount Google Drive and save there.

```python
# Run this ONCE to mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Then change SAVE_DIR in the save/load cells to:
SAVE_DIR = Path("/content/drive/MyDrive/aenpi_models")
```

After that, your workflow every new session is just:

```
1. Clone repo + pip install   (~30 seconds)
2. Run the Load cell (F2)     (~2 seconds)
3. Done — all methods work
```

No dataset download, no training, no waiting. ✅

---

### Summary of what gets saved

```
aenpi_models/
├── intent_router.joblib     # ~50–200 KB depending on vocab size
└── urdu_sentiment.joblib    # ~3–8 MB (larger vocab from 20K samples)
```

Both files together are much smaller than re-downloading the 134K-row HuggingFace dataset every time.

---
## ✅ Section E — Final Summary

In [35]:
summary = [
    # ── IntentRouter ──────────────────────────────────────────────────────────
    ("IntentRouter", "fit()",                      "Train on 25 diverse examples across 5 intents"),
    ("IntentRouter", "predict()",                  "Single text → intent + confidence"),
    ("IntentRouter", "predict_batch()",            "List of texts → list of results"),
    ("IntentRouter", "top_n()",                    "Top-N intents sorted by confidence"),
    ("IntentRouter", "score()",                    "Accuracy on labeled eval set"),
    ("IntentRouter", "add_examples() + refit()",   "Added 'payment' intent, retrained"),
    ("IntentRouter", "list_intents()",             "Returns all known labels"),
    ("IntentRouter", "example_count()",            "Per-intent training count"),
    ("IntentRouter", "edge: empty string",         "No crash — uniform low confidence"),
    ("IntentRouter", "edge: single word",          "No crash — sparse features"),
    ("IntentRouter", "edge: Urdu script",          "No crash — low conf (script mismatch)"),
    ("IntentRouter", "edge: Roman Urdu",           "No crash — works well"),
    ("IntentRouter", "edge: mixed language",       "No crash — handles code-switching"),
    ("IntentRouter", "edge: emojis/symbols",       "No crash — stripped by preprocess"),
    ("IntentRouter", "edge: predict before fit",   "RuntimeError raised correctly ✅"),
    ("IntentRouter", "edge: fit([]) empty",        "ValueError raised correctly ✅"),
    # ── UrduSentiment ────────────────────────────────────────────────────────
    ("UrduSentiment", "fit() [fixed]",             "Trained on 20K samples, col auto-detected"),
    ("UrduSentiment", "predict()",                 "Single text → Positive/Negative/Neutral"),
    ("UrduSentiment", "predict_batch()",           "Batch sentiment on 8 comments"),
    ("UrduSentiment", "score()",                   "Accuracy on 10 unseen labeled examples"),
    ("UrduSentiment", "edge: empty string",        "No crash — uniform ≈0.33 per class"),
    ("UrduSentiment", "edge: single word",         "No crash — sparse prediction"),
    ("UrduSentiment", "edge: Urdu script",         "No crash — low conf expected"),
    ("UrduSentiment", "edge: Roman Urdu",          "No crash — works well"),
    ("UrduSentiment", "edge: mixed language",      "No crash — handles code-switching"),
    ("UrduSentiment", "edge: emojis/symbols",      "No crash — stripped by preprocess"),
    ("UrduSentiment", "edge: predict before fit",  "RuntimeError raised correctly ✅"),
]

print(f"{'Module':<16} {'Method/Test':<30} {'Notes'}")
print("=" * 85)
prev = ""
for module, test, note in summary:
    if module != prev:
        print()
        prev = module
    print(f"{module:<16} {test:<30} {note}")

print(f"\n{'='*85}")
print(f"Total tests: {len(summary)}")

Module           Method/Test                    Notes

IntentRouter     fit()                          Train on 25 diverse examples across 5 intents
IntentRouter     predict()                      Single text → intent + confidence
IntentRouter     predict_batch()                List of texts → list of results
IntentRouter     top_n()                        Top-N intents sorted by confidence
IntentRouter     score()                        Accuracy on labeled eval set
IntentRouter     add_examples() + refit()       Added 'payment' intent, retrained
IntentRouter     list_intents()                 Returns all known labels
IntentRouter     example_count()                Per-intent training count
IntentRouter     edge: empty string             No crash — uniform low confidence
IntentRouter     edge: single word              No crash — sparse features
IntentRouter     edge: Urdu script              No crash — low conf (script mismatch)
IntentRouter     edge: Roman Urdu               No crash 

In [34]:
import joblib

data = joblib.load("/content/aenpi_models/intent_router.joblib")
print("Keys        :", list(data.keys()))
print("is_fitted   :", data["is_fitted"])
print("classes_    :", data["classes_"])
print("vectorizer  :", data["vectorizer"])
print("clf         :", data["clf"])
print("examples    :", len(data["_examples"]), "training pairs")

Keys        : ['vectorizer', 'clf', 'classes_', 'is_fitted', '_examples']
is_fitted   : True
classes_    : [np.str_('complaint'), np.str_('feedback'), np.str_('order'), np.str_('payment'), np.str_('refund'), np.str_('tracking')]
vectorizer  : TfidfVectorizer(analyzer='char_wb', max_features=20000, ngram_range=(2, 4),
                sublinear_tf=True)
clf         : LogisticRegression(C=10.0, max_iter=500, multi_class='multinomial')
examples    : 30 training pairs


In [37]:
import joblib

data = joblib.load("/content/aenpi_models/urdu_sentiment.joblib")
print("Keys        :", list(data.keys()))
print("is_fitted   :", data["is_fitted"])
print("classes_    :", data["classes_"])
print("vectorizer  :", data["vectorizer"])
print("clf         :", data["clf"])

Keys        : ['vectorizer', 'clf', 'classes_', 'is_fitted']
is_fitted   : True
classes_    : [np.str_('Negative'), np.str_('Neutral'), np.str_('Positive')]
vectorizer  : TfidfVectorizer(analyzer='char_wb', max_features=60000, ngram_range=(2, 4),
                sublinear_tf=True)
clf         : LogisticRegression(C=5.0, max_iter=300, multi_class='multinomial')
